# Esperimento: complexity gating per la text simplification

## RQ
Un predittore di complessità usato come filtro **aiuta o no** un semplificatore?
```

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4"  

In [ ]:
import os, gc, math, warnings
import numpy as np
import pandas as pd
from collections import Counter

import torch
from scipy.special import softmax
from transformers import (
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    AutoTokenizer,
)
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

DATA_DIR        = "/code/HLTproject_code/data"
CLS_CHECKPOINT  = "/code/HLTproject_code/task_1A_complexity_prediction/bert_cls/lr_5e-05/checkpoint-2828"
ANITA_MODEL_ID  = "swap-uniba/LLaMAntino-3-ANITA-8B-Inst-DPO-ITA"


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    # Libera memoria da eventuali run precedenti
    gc.collect()
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / 1e9
    print(f"GPU memoria libera: {free_gb:.1f} GB")
    if free_gb < 2:
        print("⚠️  Poca memoria libera — prova Kernel > Restart Kernel prima di procedere")

/data/miniconda3/envs/hlt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA A100-SXM4-40GB
GPU memoria libera: 41.9 GB


---
## Sezione 1 — Dati

In [ ]:
import numpy as np
import pandas as pd
from datasets import load_dataset

DELTA_FILTER = 0.05   # scarta semplificazioni degeneri (non riducono la leggibilità, stesso valore del task 1A)
DELTA_LABEL  = 0.10   # un input "necessita semplificazione" se dista dal gold + di questo
SEED = 42

# gli originali del test set
test = pd.read_parquet("/code/HLTproject_code/data/test.parquet")
test_orig_ids = set(test["original_sentence_idx"].unique())

# carica il dataset e tiene solo le coppie di quegli originali (con tutte le ~10 simp)
full = load_dataset("mpapucci/impacts", "wikipedia", split="train").to_pandas()
full = full[full["original_sentence_idx"].isin(test_orig_ids)].copy()
print(f"originali di test: {len(test_orig_ids)} | righe (coppie) caricate: {len(full)}")

originali di test: 7539 | righe (coppie) caricate: 55832


In [ ]:
rows = []
for oidx, grp in full.groupby("original_sentence_idx"):
    # semplificazioni valide 
    valid = grp[grp["original_all"] - grp["simplification_all"] > DELTA_FILTER]
    if len(valid) == 0:
        continue                                    # nessuna semplificazione utile -> scarta

    # GOLD = la semplificazione valida piu' semplice (READ-IT minimo)
    gold_row   = valid.loc[valid["simplification_all"].idxmin()]
    gold_text  = gold_row["simplification"]
    gold_score = gold_row["simplification_all"]

    # INPUT = originale + tutte le altre semplificazioni valide 
    inputs = [(grp.iloc[0]["original_text"], grp.iloc[0]["original_all"])]   # l'originale
    for _, r in valid.drop(gold_row.name).iterrows():                        # le intermedie
        inputs.append((r["simplification"], r["simplification_all"]))

    for text, score in inputs:
        gap   = score - gold_score                  # quanto e' ancora lontano dal gold
        label = int(gap > DELTA_LABEL)              # 1 = da semplificare, 0 = gia' semplice
        rows.append({
            "original_sentence_idx": oidx,
            "input_text":  text,
            "input_readit": score,
            "gold_text":   gold_text,
            "gold_readit": gold_score,
            "gap_to_gold": gap,
            "true_label":  label,
        })

gating = pd.DataFrame(rows)

In [5]:
print(f"originali usati : {gating['original_sentence_idx'].nunique()}")
print(f"input totali    : {len(gating)}")
print(gating['true_label'].value_counts())
print(f"  label 1 (da semplificare): {(gating['true_label']==1).mean()*100:.1f}%")
print(f"  label 0 (gia' semplice)  : {(gating['true_label']==0).mean()*100:.1f}%")
gating.head()

originali usati : 7539
input totali    : 48472
true_label
1    31642
0    16830
Name: count, dtype: int64
  label 1 (da semplificare): 65.3%
  label 0 (gia' semplice)  : 34.7%


,original_sentence_idx,input_text,input_readit,gold_text,gold_readit,gap_to_gold,true_label
0,10,"Prodotto dalla BBC, il film esce solo nel 1998...",0.772007,"Prodotto dalla BBC nel 1998, il film ha ricevu...",0.213621,0.558386,1
1,10,"Prodotto dalla BBC, il film è stato pubblicato...",0.411796,"Prodotto dalla BBC nel 1998, il film ha ricevu...",0.213621,0.198175,1
2,10,"Un produttore della BBC, il film è stato rilas...",0.369291,"Prodotto dalla BBC nel 1998, il film ha ricevu...",0.213621,0.155671,1
3,10,"Nel 1998, il film è stato rilasciato e ha rice...",0.378216,"Prodotto dalla BBC nel 1998, il film ha ricevu...",0.213621,0.164595,1
4,10,"Un produttore della BBC, il film è stato rilas...",0.351118,"Prodotto dalla BBC nel 1998, il film ha ricevu...",0.213621,0.137497,1


In [6]:
train_ids = set(pd.read_parquet("/code/HLTproject_code/data/train.parquet")["original_sentence_idx"])
assert len(train_ids & set(gating["original_sentence_idx"])) == 0, "LEAKAGE!"
print("ok: nessun originale di gating era nel train")

ok: nessun originale di gating era nel train


In [7]:
from sklearn.model_selection import train_test_split

N_SAMPLE = 10_000   # input totali da valutare (regola in base al tempo di ANITA)

gating_sample, _ = train_test_split(
    gating,
    train_size=N_SAMPLE,
    stratify=gating["true_label"],   # mantiene il 65/35
    random_state=SEED,
)
gating_sample = gating_sample.reset_index(drop=True)

print(f"campione: {len(gating_sample)} input")
print(gating_sample["true_label"].value_counts())
print(f"  label 1: {(gating_sample['true_label']==1).mean()*100:.1f}%")
print(f"  label 0: {(gating_sample['true_label']==0).mean()*100:.1f}%")

campione: 10000 input
true_label
1    6528
0    3472
Name: count, dtype: int64
  label 1: 65.3%
  label 0: 34.7%


---
## Sezione 2 — BERTino: predittore di complessità

BERTino decide per ogni frase se è complessa o semplice.
La soglia viene calibrata sul val set (stesso approccio del training).

In [8]:
val  = pd.read_parquet(os.path.join(DATA_DIR, "val.parquet"))

In [9]:
cls_tok   = AutoTokenizer.from_pretrained(CLS_CHECKPOINT)
cls_model = AutoModelForSequenceClassification.from_pretrained(CLS_CHECKPOINT).to(device).eval()


def predict_probs(texts, batch_size=64):
    """
    Inferenza BERTino senza Trainer.
    Processa un batch alla volta e sposta subito i risultati su CPU,
    così non accumula nulla in GPU memory.
    """
    all_probs = []
    for k in range(0, len(texts), batch_size):
        batch = texts[k : k + batch_size]
        enc   = cls_tok(
            batch, truncation=True, max_length=128,
            padding=True, return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            logits = cls_model(**enc).logits          # (B, 2) su GPU
        probs = softmax(logits.cpu().float().numpy(), axis=-1)[:, 1]
        all_probs.extend(probs.tolist())
        del enc, logits                               # libera subito la memoria GPU
    return np.array(all_probs)


# Calibrazione soglia su val: cerca il threshold che massimizza F1 su val
from sklearn.metrics import f1_score
val_texts  = val["text"].tolist()
val_labels = val["is_original"].astype(int).values

val_probs = predict_probs(val_texts)
cands = np.quantile(val_probs, np.linspace(0.1, 0.9, 81))
thr   = max(cands, key=lambda t: f1_score(val_labels, (val_probs > t).astype(int)))
print(f"Soglia calibrata su val: {thr:.4f}")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 8032.11it/s]


Soglia calibrata su val: 0.5427


In [10]:
input_probs = predict_probs(gating_sample["input_text"].tolist())
gating_sample["pred_gate"] = (input_probs > thr).astype(int)
print(gating_sample["pred_gate"].value_counts())

pred_gate
0    6603
1    3397
Name: count, dtype: int64


In [11]:
def quadrante(r):
    t, p = r["true_label"], r["pred_gate"]
    return ("TP" if t==1 and p==1 else
            "FN" if t==1 and p==0 else
            "TN" if t==0 and p==0 else "FP")

gating_sample["quadrante"] = gating_sample.apply(quadrante, axis=1)
print(gating_sample["quadrante"].value_counts())

quadrante
FN    3680
TN    2923
TP    2848
FP     549
Name: count, dtype: int64


In [12]:
import pandas as pd
print(pd.crosstab(gating_sample["true_label"], gating_sample["pred_gate"],
                  rownames=["true"], colnames=["gate"]))

gate     0     1
true            
0     2923   549
1     3680  2848


In [13]:
gating_sample.to_parquet(os.path.join(DATA_DIR, "gating_sample.parquet"))
print("salvato:", len(gating_sample), "input con pred_gate e quadrante")

salvato: 10000 input con pred_gate e quadrante


In [ ]:
# gating_sample[:20].to_parquet(os.path.join(DATA_DIR, "gating_sample.parquet"))
# print("salvato:", len(gating_sample), "input con pred_gate e quadrante")

In [14]:
df = pd.read_parquet("/code/HLTproject_code/task_2_simplification_generation_gating/gating_data/gating_with_outputs.parquet")

In [15]:
df['anita_output'][4]

"Fu giudicato dalla Corte militare a Metz, dal 9 al 13 novembre 1903, per offese all'esercito tedesco, e in quel tempo il suo lavoro fu sospeso."

In [16]:
len(df)

10000

---
## Sezione 4 — Valutazione

In [17]:
import numpy as np
import pandas as pd
import spacy
from difflib import SequenceMatcher
from sklearn.metrics import precision_recall_fscore_support
from sacrebleu.metrics import TER
import torch
import numpy as np
from captum.attr import LayerIntegratedGradients
import unicodedata
from difflib import SequenceMatcher

In [18]:
import pandas as pd
df = pd.read_parquet("/code/HLTproject_code/task_2_simplification_generation_gating/gating_data/gating_with_outputs.parquet")

def overlap(a, b):
    sa, sb = set(a.lower().split()), set(b.lower().split())
    return len(sa & sb) / max(len(sa), 1)

df["ov"] = [overlap(i, g) for i, g in zip(df["input_text"], df["gold_text"])]

df_clean = df[df["ov"] >= 0.2].copy()
print(f"da {len(df)} a {len(df_clean)} coppie")
print(df_clean["quadrante"].value_counts())   # controlla che i 4 quadranti restino popolati

da 10000 a 8306 coppie
quadrante
FN    2811
TN    2624
TP    2354
FP     517
Name: count, dtype: int64


In [20]:
fn = df_clean[df_clean["quadrante"]=="FN"].sample(3, random_state=1)
for _, r in fn.iterrows():
    print("INPUT:", r["input_text"][:150])
    print("GOLD :", r["gold_text"][:150])
    print("ANITA:", r["anita_output"][:150])
    print("overlap:", round(r["ov"],2), "\n")

INPUT: Il neopaganesimo, il giudeopaganesimo, il sincretismo cristiano-pagano, l'indopaganesimo e il gaianesimo sono movimenti eclettici che hanno influenzat
GOLD : La New Age ha influenzato movimenti particolarmente eclettici come il giudeopaganesimo, il sincretismo cristiano-pagano, l'indopaganesimo e il gaianes
ANITA: Alcuni movimenti, come il neopaganesimo e gli altri, hanno influito sulla New Age.
overlap: 0.71 

INPUT: Nel 2012, il Ludogorets ha completato il "treble" vincendo anche la Supercoppa di Bulgaria.
GOLD : Ludogorets ha completato il treble nazionale nel 2012
ANITA: Nel 2012, il Ludogorets vinse tutto, incluso la Supercoppa di Bulgaria.
overlap: 0.38 

INPUT: Si è laureato nel 1899 e ha partecipato ai Giochi della II Olimpiade di Parigi del 1900.
GOLD : Nel 1899, partecipò ai Giochi della II Olimpiade di Parigi.
ANITA: Si è laureato nel 1899 e ha preso parte ai Giochi olimpici di Parigi del 1900.
overlap: 0.41 



In [24]:
import numpy as np
import pandas as pd
from evaluate import load
sari = load("sari")

import numpy as np
from evaluate import load
sari = load("sari")

_ter = TER()

def word_edit_distance(a, b, sub_cost=1.0):
    """Levenshtein a livello di parola"""
    n, m = len(a), len(b)
    dp = [[0.0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): dp[i][0] = i
    for j in range(m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+sub_cost)
    return dp[n][m]

def precompute_scores(dfx):
    """Calcola SARI/ED/TER per-frase UNA volta, per entrambe le condizioni.
       Ritorna il df con 6 colonne nuove: {sari,ed,ter}_{A,B}."""
    dfx = dfx.copy()
    dfx["out_A"] = dfx["anita_output"]
    dfx["out_B"] = np.where(dfx["pred_gate"] == 1, dfx["anita_output"], dfx["input_text"])

    src  = dfx["input_text"].tolist()
    gold = dfx["gold_text"].tolist()

    for cond in ["A", "B"]:
        out = dfx[f"out_{cond}"].tolist()
        # SARI: una chiamata per frase (necessario per avere il punteggio per-frase),
        #       ma calcolata UNA sola volta e messa in colonna
        dfx[f"sari_{cond}"] = [sari.compute(sources=[s], predictions=[o], references=[[g]])["sari"]
                               for s, o, g in zip(src, out, gold)]
        dfx[f"ed_{cond}"]   = [word_edit_distance(o.split(), g.split(), 1.5)
                               for o, g in zip(out, gold)]
        dfx[f"ter_{cond}"]  = [_ter.sentence_score(o, [g]).score
                               for o, g in zip(out, gold)]
    return dfx

def tabella(dfx):
    """Aggrega le colonne pre-calcolate per quadrante + totale (istantaneo)."""
    cols = ["sari_A","ed_A","ter_A","sari_B","ed_B","ter_B"]
    righe = []
    for q in ["TP", "FN", "TN", "FP"]:
        sub = dfx[dfx["quadrante"] == q]
        if len(sub) == 0: continue
        righe.append([q, len(sub)] + sub[cols].mean().tolist())
    righe.append(["TOTALE", len(dfx)] + dfx[cols].mean().tolist())
    return pd.DataFrame(righe, columns=["quadrante","n",
                        "SARI_A","ED_A","TER_A","SARI_B","ED_B","TER_B"])

# --- USO: il calcolo pesante gira UNA volta ---
df_scored = precompute_scores(df_clean)     # <-- lento, ma una tantum (qualche minuto)
print(tabella(df_scored).round(2))          # <-- istantaneo, e riusabile

  quadrante     n  SARI_A   ED_A  TER_A  SARI_B   ED_B   TER_B
0        TP  2354   36.78  23.17  96.91   36.78  23.17   96.91
1        FN  2811   36.41  23.59  98.92   50.05  22.89  104.90
2        TN  2624   31.98  19.30  85.28   57.44  13.13   63.97
3        FP   517   30.74  19.73  83.06   30.74  19.73   83.06
4    TOTALE  8306   34.76  21.87  93.05   47.42  19.69   88.35
